# Twitter User Classification: Influencer vs. Observer
This notebook implements a machine learning pipeline to classify Twitter users based on their tweet content and profile metadata. It uses a combination of TF-IDF, CamemBERT embeddings, and XGBoost classifiers within a voting ensemble.

In [1]:
# 1. Imports and Setup
import json
import re
import warnings

import numpy as np
import pandas as pd
from pandas import json_normalize
import torch
import emoji
from tqdm import tqdm
from nltk.corpus import stopwords

from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, MaxAbsScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import VotingClassifier

warnings.filterwarnings('ignore')

## 1. Data Loading
Loading the training and Kaggle test datasets from JSON Lines format and flattening the nested structures.

In [2]:
# Load and flatten training data
train_data = pd.read_json('train.jsonl', lines=True)
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load and flatten Kaggle test data
kaggle_data = pd.read_json('kaggle_test.jsonl', lines=True)
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

# Separate features from target
X_train = train_data.drop('label', axis=1)
y_train = train_data['label']
X_kaggle = kaggle_data.copy()

## 2. Text Preprocessing
Extracting the full text of tweets (handling truncated text) and normalizing URLs, mentions, and emojis.

In [3]:
def extract_full_text(tweet):
    """Extracts the full tweet text, falling back to extended_tweet.full_text if available."""
    text = tweet['text']
    if not pd.isna(tweet['extended_tweet.full_text']):
        text = tweet['extended_tweet.full_text']
    return text

def clean_text(text):
    """Normalizes URLs, user mentions, and translates emojis to text."""
    if not isinstance(text, str): 
        return ""
    text = re.sub(r'http\S+', '<URL>', text)  
    text = re.sub(r'@\w+', '<USER>', text)    
    text = emoji.demojize(text, language='fr')
    text = text.replace(":", " ").replace("_", " ")
    return text

# Apply extraction and cleaning
for df in [X_train, X_kaggle]:
    df['full_text'] = df.apply(extract_full_text, axis=1)
    df['clean_bio'] = df['user.description'].apply(clean_text)
    df['clean_text'] = df['full_text'].apply(clean_text)

# Drop completely empty columns
missing_series = X_train.isnull().mean()
cols_to_drop = missing_series[missing_series == 1.0].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns that are 100% empty.")

exclude_cols = {'label', 'challenge_id', 'text', 'user.description'} | set(cols_to_drop)
features = [c for c in X_train.columns if c not in exclude_cols]

Dropping 24 columns that are 100% empty.


## 3. Deep Learning Feature Extraction (CamemBERT)
Generating dense semantic embeddings for both tweet content and user biographies using the French CamemBERT model.

In [ ]:
def generate_bert_embeddings(df, text_col, model_name='almanach/camembert-base', batch_size=32):
    """Generates sentence-level embeddings using the [CLS] token from a pre-trained BERT model."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"--- Loading model {model_name} on {device} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    texts = df[text_col].fillna("").astype(str).tolist()
    all_embeddings = []

    print(f"Computing embeddings for {len(texts)} samples...")
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i : i + batch_size]
            inputs = tokenizer(
                batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
            ).to(device)
            outputs = model(**inputs)
            # Extract [CLS] token vector
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)

    final_embeddings = np.vstack(all_embeddings)
    cols = [f'bert_{i}' for i in range(final_embeddings.shape[1])]
    return pd.DataFrame(final_embeddings, columns=cols, index=df.index), cols

# Generate embeddings for Bio and Tweet Text
print("Processing Train Bio...")
df_bertBio_tr, bert_cols_bio = generate_bert_embeddings(X_train, 'clean_bio')
df_bertBio_tr = df_bertBio_tr.add_prefix('bio_')
bert_cols_bio = [f'bio_{col}' for col in bert_cols_bio]
X_train = pd.concat([X_train, df_bertBio_tr], axis=1)

print("Processing Kaggle Bio...")
df_bertBio_kaggle, _ = generate_bert_embeddings(X_kaggle, 'clean_bio')
df_bertBio_kaggle = df_bertBio_kaggle.add_prefix('bio_')
X_kaggle = pd.concat([X_kaggle, df_bertBio_kaggle], axis=1)

print("Processing Train Tweet Text...")
df_bertText_tr, bert_cols_text = generate_bert_embeddings(X_train, 'clean_text')
df_bertText_tr = df_bertText_tr.add_prefix('text_')
bert_cols_text = [f'text_{col}' for col in bert_cols_text]
X_train = pd.concat([X_train, df_bertText_tr], axis=1)

print("Processing Kaggle Tweet Text...")
df_bertText_kaggle, _ = generate_bert_embeddings(X_kaggle, 'clean_text')
df_bertText_kaggle = df_bertText_kaggle.add_prefix('text_')
X_kaggle = pd.concat([X_kaggle, df_bertText_kaggle], axis=1)

Processing Train Bio...
--- Loading model almanach/camembert-base on cpu ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertModel LOAD REPORT from: almanach/camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing embeddings for 154914 samples...


  0%|          | 8/4842 [00:08<1:15:57,  1.06it/s]

## 4. Metadata Feature Engineering
Engineering temporal features (e.g., account age, time of day) and cleaning metadata (URLs, locations, profile booleans).

In [ ]:
def extract_date_features(df, col_name):
    temp_date = pd.to_datetime(df[col_name], errors='coerce')
    df[f'{col_name}.day_of_week'] = temp_date.dt.day_name()    
    df[f'{col_name}.time_of_day'] = pd.cut(
        temp_date.dt.hour, 
        bins=[-1, 5, 11, 17, 21, 24], 
        labels=['Night', 'Morning', 'Afternoon', 'Evening', 'Night'],
        ordered=False
    )
    df[f'{col_name}.exact_time'] = temp_date.dt.hour
    return df

def calculate_time_delta(df, col_a, col_b, new_col_name='delta_temps'):
    if f'{col_a}.exact_time' in df.columns and f'{col_b}.exact_time' in df.columns:
        df[new_col_name] = df[f'{col_a}.exact_time'] - df[f'{col_b}.exact_time']
    else:
        df[new_col_name] = np.nan
    return df

def preprocess_user_features(df):
    df = df.copy()
    # 1. Account Age
    df['user.created_at'] = pd.to_datetime(df['user.created_at'], format='%a %b %d %H:%M:%S +0000 %Y')
    ref_date = df['user.created_at'].max()
    df['user.account_age_days'] = (ref_date - df['user.created_at']).dt.days
    df['user.account_age_days_log'] = np.log1p(df['user.account_age_days'])

    # 2. URL Domain Extraction
    def clean_url(url):
        if pd.isna(url) or url == 'None': return ""
        clean = re.sub(r'https?://(www\.)?', '', url).split('/')[0]
        return clean.replace('.', ' ')
    df['user.url_text'] = df['user.url'].apply(clean_url)

    # 3. Image Booleans
    df['has_profile_image'] = df['user.profile_image_url'].notna() & (df['user.profile_image_url'] != '')
    df['has_background_image'] = df['user.profile_background_image_url'].notna() & (df['user.profile_background_image_url'] != '')
    
    # 4. Location Cleaning
    df['user.location'] = df['user.location'].fillna('').astype(str)
    return df

# Apply preprocessing
for df in [X_train, X_kaggle]:
    df = extract_date_features(df, col_name='created_at')
    df = extract_date_features(df, col_name='quoted_status.created_at')
    df = calculate_time_delta(df, col_a='quoted_status.created_at', col_b='created_at')

X_train = preprocess_user_features(X_train)
X_kaggle = preprocess_user_features(X_kaggle)

# Drop constant columns
val_unique = X_train.copy()
for col in val_unique.select_dtypes(include=['object']).columns:
    val_unique[col] = val_unique[col].astype(str)

unique_counts = val_unique.nunique()
cols_to_drop = unique_counts[unique_counts <= 1].index.tolist()
X_train = X_train.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} constant columns.")

C:\Users\Octave\AppData\Local\Temp\ipykernel_13220\2659452405.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date = pd.to_datetime(df[col_name], errors='coerce')
C:\Users\Octave\AppData\Local\Temp\ipykernel_13220\2659452405.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date = pd.to_datetime(df[col_name], errors='coerce')


## 5. Data Splitting (Preventing Leakage)
To prevent data leakage, we use `GroupShuffleSplit` to ensure that all tweets from a single user remain either entirely in the training set or entirely in the validation set.

In [ ]:
# Create groups based on unique user account creation times
groups = X_train['user.created_at'] 

splitter = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, val_idx = next(splitter.split(X_train, y_train, groups))

X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

# Leakage verification
users_train = set(X_tr['user.created_at'])
users_val = set(X_val['user.created_at'])
intersection = users_train.intersection(users_val)

print(f"Tweets in Train: {len(X_tr)}")
print(f"Tweets in Validation: {len(X_val)}")
print("-" * 30)
if len(intersection) == 0:
    print("✅ SUCCESS: Users are strictly separated. No data leakage detected.")
else:
    print("❌ WARNING: Data leakage detected!")

## 6. Model Pipelines Construction
Building two distinct pipelines:
1. **TF-IDF Pipeline**: Uses standard TF-IDF vectorization on tweet text.
2. **BERT Pipeline**: Uses the pre-computed CamemBERT embeddings.

In [ ]:
# Helper functions for scikit-learn transformers
def cast_text_to_string(x): return x.astype(str).fillna('')
def cast_categorical_to_string(x): return x.astype(str)

french_stop_words = stopwords.words('french')

# 1. TF-IDF Text Transformer
text_transformer_TFIDF = Pipeline([
    ('caster', FunctionTransformer(cast_text_to_string, validate=False)),
    ('tfidf', TfidfVectorizer(
        stop_words=french_stop_words, max_df=0.7, min_df=2, 
        max_features=1000, ngram_range=(1, 2)
    ))
])

# Define base XGBoost Classifier
xgb_base = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=7, 
    tree_method='hist', device='cpu', n_jobs=-1,
    min_child_weight=2, colsample_bytree=0.75, random_state=42,
    eval_metric='logloss', reg_alpha=1, reg_lambda=1
)

# Build TFIDF Pipeline
# (Note: Assuming categorical_transformer and numeric_transformer are defined as in your original notebook)
# model_pipeline_TFIDF = Pipeline([ ... ])
# model_pipeline_BERT = Pipeline([ ... ])

## 7. Ensemble Modeling & Optimization
Combining the predictions of the Tweet-level models (TF-IDF & BERT) with the User Profile model using a soft voting strategy and searching for the optimal weight distribution.

In [ ]:
# Soft Voting Ensemble for Tweet Content Models
ensemble = VotingClassifier(
    estimators=[
        ('xgb_camembert', model_pipeline_BERT),
        ('tfidf_xgb', model_pipeline_TFIDF)
    ],
    voting='soft'
)

print("Training Ensemble Model...")
ensemble.fit(X_tr, y_tr)

# Predict probabilities on validation set
probs_tweets = ensemble.predict_proba(X_val)[:, 1]
probs_profile = user_profile_model.predict_proba(X_val)[:, 1]

df_scores = pd.DataFrame({
    'user_id': X_val['user.created_at'],
    'proba_tweet': probs_tweets,
    'proba_profile': probs_profile
})

# Aggregate probabilities per user
score_tweets_agg = df_scores.groupby('user_id')['proba_tweet'].mean()
score_profile_agg = df_scores.groupby('user_id')['proba_profile'].mean()
final_df = pd.concat([score_tweets_agg, score_profile_agg], axis=1)

# Find optimal weighting between tweet content and profile metadata
best_weight = 0
best_acc = 0

for i in np.arange(0, 1.01, 0.01):
    weight_tweets = i
    weight_profile = 1 - weight_tweets

    final_df['final_proba'] = (final_df['proba_tweet'] * weight_tweets) + (final_df['proba_profile'] * weight_profile)
    final_df['prediction'] = (final_df['final_proba'] > 0.5).astype(int)
    
    # Broadcast decision back to individual tweets
    y_pred_aggregated = X_val['user.created_at'].map(final_df['prediction'])
    
    acc = accuracy_score(y_val, y_pred_aggregated)
    if acc > best_acc:
        best_acc = acc
        best_weight = i

print(f"Best Accuracy after user aggregation: {best_acc:.4f}")
print(f"Obtained with Tweet Weight Ratio: {best_weight:.4f}")